# Temporary Market Impact Parameter Estimation

This notebook estimates temporary market impact on a trade-count clock.

- Collapse multi-hit fills into parent trades by `(exchange, floored timestamp, side)`.
- Compute signed markouts: `y = sgn * (p_fwd / p - 1)`, where `p_fwd` is `M` trades ahead.
- Fit square-root and power-law impact forms by date and volume quantile cohort.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd

from impact_estimation.impact_config import ImpactConfig
from impact_estimation.io import find_trade_files, read_trades_parquet, explore_trades
from impact_estimation.impact_core import collapse_multihit_trades, run_exchange_and_pooled, robustness_sweep, infer_side_signs, add_markouts, trim_outliers_and_bin
from impact_estimation.plots import plot_fit, save_figure

In [ ]:
repo_root = Path('.').resolve()
outdir = repo_root / 'outputs'
outdir.mkdir(parents=True, exist_ok=True)
figdir = outdir / 'figures'
figdir.mkdir(parents=True, exist_ok=True)

cfg = ImpactConfig(M=50, ts_floor='1ms', min_n=1000)
files = find_trade_files(repo_root)
print(f'Found {len(files)} parquet trade files')
files[:5]

In [ ]:
if files:
    demo = read_trades_parquet(files[0], max_rows=200000, sample_frac=None)
    print('One-file demo shape:', demo.shape)
    explore_trades(demo)
else:
    print('No files found; using synthetic fallback for demonstration.')

In [ ]:
if files:
    chunks = [read_trades_parquet(f, cfg.max_rows_per_file, cfg.sample_frac) for f in files]
    trades = pd.concat(chunks, ignore_index=True)
else:
    n = 30000
    trades = pd.DataFrame({
        'ts': pd.date_range('2024-01-01', periods=n, freq='ms', tz='UTC'),
        'exchange': ['SYNX' if i % 2 == 0 else 'SYNY' for i in range(n)],
        'side': ['BUY' if i % 3 == 0 else 'SELL' for i in range(n)],
        'qty': 1 + (pd.Series(range(n)) % 150).to_numpy(),
        'price': 100 + (pd.Series(range(n)).to_numpy() * 1e-4),
    })

parents = collapse_multihit_trades(trades, cfg.ts_floor)
results = run_exchange_and_pooled(parents, cfg)
print('Parent rows:', len(parents))
print('Result rows:', len(results))
results.head()

In [ ]:
results.to_parquet(outdir / 'impact_results.parquet', index=False)
results.to_csv(outdir / 'impact_results.csv', index=False)
print('Saved base results tables.')

In [ ]:
if not parents.empty and not results.empty:
    side_map = infer_side_signs(parents, cfg.M)
    markouts = add_markouts(parents, cfg.M, side_map)
    binned = trim_outliers_and_bin(markouts, cfg.q_outlier, cfg.q_bins)
    sample = results.iloc[0]
    dplot = binned[binned['qbin'] == sample['qbin']]
    if not dplot.empty:
        fig = plot_fit(dplot, sample['mu0'], sample['mu1'], sample['mu2'], f"{sample['label']} {sample['date']} {sample['qbin']}")
        fig_path = save_figure(fig, figdir, f"fit_{sample['label']}_{sample['date']}_{sample['qbin']}")
        print('Saved figure:', fig_path)

In [ ]:
rob = robustness_sweep(parents, cfg)
rob.to_parquet(outdir / 'impact_robustness.parquet', index=False)
rob.to_csv(outdir / 'impact_robustness.csv', index=False)
print('Robustness rows:', len(rob))
if not rob.empty:
    summary = rob.groupby(['label','M','q_outlier'], observed=True)['mu2'].median().reset_index()
    summary = summary.sort_values(['label','M','q_outlier'])
    summary.head(20)